# Open-Meteo API Test — CropSage Texas

## Demonstration location

This notebook uses a configurable rural coordinate west of Plainview in Hale County, Texas:

- Latitude: 34.1800
- Longitude: -101.7600
- CropSage region: Texas Plains
- Location type: agricultural field area, not a city-centre weather point

## Objectives

1. Confirm that the public Open-Meteo endpoints work without an API key.
2. Test city/postal-code geocoding separately from farm-coordinate weather lookup.
3. Resolve the farm's IANA time zone automatically.
4. retrieve current modeled soil moisture and soil temperature by depth.
5. retrieve the previous seven complete days of rainfall.
6. retrieve a seven-day rainfall, temperature, ET₀ and VPD forecast.
7. detect missing fields and null values.
8. calculate simple operational indicators without presenting them as exact irrigation prescriptions.
9. verify API validation-error behaviour.

## Important limitation

Open-Meteo soil moisture and weather values are model estimates for a grid cell. They are not measurements from the selected farm. SSURGO and farmer observations will provide separate soil evidence.

In [1]:
from datetime import timedelta

import pandas as pd
import requests

print("Imports successful")

Imports successful


In [2]:
# Rural agricultural coordinate west of Plainview, Hale County, Texas.
FARM_LATITUDE = 34.1800
FARM_LONGITUDE = -101.7600

LOCATION_QUERY = "Plainview, Texas"
COUNTRY_CODE = "US"

PAST_DAYS = 7
FORECAST_DAYS = 7
TIMEOUT_SECONDS = 90

GEOCODING_URL = "https://geocoding-api.open-meteo.com/v1/search"
FORECAST_URL = "https://api.open-meteo.com/v1/forecast"

print("Farm coordinate:", FARM_LATITUDE, FARM_LONGITUDE)

Farm coordinate: 34.18 -101.76


## 1. Location-name geocoding test

Open-Meteo geocoding supports cities and postal codes. CropSage will still need coordinate/map-pin input or a dedicated address geocoder for exact rural farms.

In [3]:
geocoding_params = {
    "name": LOCATION_QUERY,
    "count": 10,
    "language": "en",
    "format": "json",
    "countryCode": COUNTRY_CODE,
}

geocoding_response = requests.get(
    GEOCODING_URL,
    params=geocoding_params,
    timeout=TIMEOUT_SECONDS,
)

print("HTTP status:", geocoding_response.status_code)
print("Request URL:")
print(geocoding_response.url)

if not geocoding_response.ok:
    print(geocoding_response.text)

geocoding_response.raise_for_status()
geocoding_data = geocoding_response.json()

geocoding_results = geocoding_data.get("results", [])

texas_candidates = [
    result
    for result in geocoding_results
    if result.get("country_code") == "US"
    and result.get("admin1") == "Texas"
]

if not texas_candidates:
    raise ValueError("No Texas geocoding result was returned.")

geocoded_location = texas_candidates[0]

geocoding_summary = pd.Series({
    "name": geocoded_location.get("name"),
    "admin1": geocoded_location.get("admin1"),
    "admin2": geocoded_location.get("admin2"),
    "latitude": geocoded_location.get("latitude"),
    "longitude": geocoded_location.get("longitude"),
    "elevation_m": geocoded_location.get("elevation"),
    "timezone": geocoded_location.get("timezone"),
})

geocoding_summary

HTTP status: 200
Request URL:
https://geocoding-api.open-meteo.com/v1/search?name=Plainview%2C+Texas&count=10&language=en&format=json&countryCode=US


name                 Plainview
admin1                   Texas
admin2                    Hale
latitude              34.18479
longitude           -101.70684
elevation_m             1025.0
timezone       America/Chicago
dtype: object

## 2. Farm-coordinate weather request

The weather request uses the rural farm coordinate, not the city-centre coordinate returned by geocoding.

In [4]:
CURRENT_VARIABLES = [
    "temperature_2m",
    "relative_humidity_2m",
    "precipitation",
    "rain",
    "weather_code",
    "soil_temperature_0cm",
    "soil_temperature_6cm",
    "soil_temperature_18cm",
    "soil_temperature_54cm",
    "soil_moisture_0_to_1cm",
    "soil_moisture_1_to_3cm",
    "soil_moisture_3_to_9cm",
    "soil_moisture_9_to_27cm",
    "soil_moisture_27_to_81cm",
]

HOURLY_VARIABLES = [
    "temperature_2m",
    "relative_humidity_2m",
    "dew_point_2m",
    "precipitation",
    "rain",
    "precipitation_probability",
    "et0_fao_evapotranspiration",
    "vapour_pressure_deficit",
    "soil_temperature_0cm",
    "soil_temperature_6cm",
    "soil_temperature_18cm",
    "soil_temperature_54cm",
    "soil_moisture_0_to_1cm",
    "soil_moisture_1_to_3cm",
    "soil_moisture_3_to_9cm",
    "soil_moisture_9_to_27cm",
    "soil_moisture_27_to_81cm",
]

DAILY_VARIABLES = [
    "temperature_2m_max",
    "temperature_2m_min",
    "precipitation_sum",
    "rain_sum",
    "precipitation_probability_max",
    "et0_fao_evapotranspiration",
]

forecast_params = {
    "latitude": FARM_LATITUDE,
    "longitude": FARM_LONGITUDE,
    "current": ",".join(CURRENT_VARIABLES),
    "hourly": ",".join(HOURLY_VARIABLES),
    "daily": ",".join(DAILY_VARIABLES),
    "past_days": PAST_DAYS,
    "forecast_days": FORECAST_DAYS,
    "timezone": "auto",
    "temperature_unit": "celsius",
    "precipitation_unit": "mm",
}

forecast_response = requests.get(
    FORECAST_URL,
    params=forecast_params,
    timeout=TIMEOUT_SECONDS,
)

print("HTTP status:", forecast_response.status_code)
print("Request URL:")
print(forecast_response.url)

if not forecast_response.ok:
    print(forecast_response.text)

forecast_response.raise_for_status()
forecast_data = forecast_response.json()

HTTP status: 200
Request URL:
https://api.open-meteo.com/v1/forecast?latitude=34.18&longitude=-101.76&current=temperature_2m%2Crelative_humidity_2m%2Cprecipitation%2Crain%2Cweather_code%2Csoil_temperature_0cm%2Csoil_temperature_6cm%2Csoil_temperature_18cm%2Csoil_temperature_54cm%2Csoil_moisture_0_to_1cm%2Csoil_moisture_1_to_3cm%2Csoil_moisture_3_to_9cm%2Csoil_moisture_9_to_27cm%2Csoil_moisture_27_to_81cm&hourly=temperature_2m%2Crelative_humidity_2m%2Cdew_point_2m%2Cprecipitation%2Crain%2Cprecipitation_probability%2Cet0_fao_evapotranspiration%2Cvapour_pressure_deficit%2Csoil_temperature_0cm%2Csoil_temperature_6cm%2Csoil_temperature_18cm%2Csoil_temperature_54cm%2Csoil_moisture_0_to_1cm%2Csoil_moisture_1_to_3cm%2Csoil_moisture_3_to_9cm%2Csoil_moisture_9_to_27cm%2Csoil_moisture_27_to_81cm&daily=temperature_2m_max%2Ctemperature_2m_min%2Cprecipitation_sum%2Crain_sum%2Cprecipitation_probability_max%2Cet0_fao_evapotranspiration&past_days=7&forecast_days=7&timezone=auto&temperature_unit=celsius

In [5]:
print("Top-level fields:", list(forecast_data.keys()))
print("Returned grid coordinate:", (
    forecast_data.get("latitude"),
    forecast_data.get("longitude"),
))
print("Grid elevation:", forecast_data.get("elevation"), "m")
print("Timezone:", forecast_data.get("timezone"))
print("Timezone abbreviation:", forecast_data.get("timezone_abbreviation"))
print("UTC offset seconds:", forecast_data.get("utc_offset_seconds"))

print("\nCurrent units:")
print(forecast_data.get("current_units", {}))

print("\nDaily units:")
print(forecast_data.get("daily_units", {}))

Top-level fields: ['latitude', 'longitude', 'generationtime_ms', 'utc_offset_seconds', 'timezone', 'timezone_abbreviation', 'elevation', 'current_units', 'current', 'hourly_units', 'hourly', 'daily_units', 'daily']
Returned grid coordinate: (34.18354, -101.76819)
Grid elevation: 1034.0 m
Timezone: America/Chicago
Timezone abbreviation: GMT-5
UTC offset seconds: -18000

Current units:
{'time': 'iso8601', 'interval': 'seconds', 'temperature_2m': '°C', 'relative_humidity_2m': '%', 'precipitation': 'mm', 'rain': 'mm', 'weather_code': 'wmo code', 'soil_temperature_0cm': '°C', 'soil_temperature_6cm': '°C', 'soil_temperature_18cm': '°C', 'soil_temperature_54cm': '°C', 'soil_moisture_0_to_1cm': 'm³/m³', 'soil_moisture_1_to_3cm': 'm³/m³', 'soil_moisture_3_to_9cm': 'm³/m³', 'soil_moisture_9_to_27cm': 'm³/m³', 'soil_moisture_27_to_81cm': 'm³/m³'}

Daily units:
{'time': 'iso8601', 'temperature_2m_max': '°C', 'temperature_2m_min': '°C', 'precipitation_sum': 'mm', 'rain_sum': 'mm', 'precipitation_pr

## 3. Current modeled soil profile

In [6]:
current_values = forecast_data["current"]
current_units = forecast_data.get("current_units", {})

soil_moisture_fields = {
    "0-1 cm": "soil_moisture_0_to_1cm",
    "1-3 cm": "soil_moisture_1_to_3cm",
    "3-9 cm": "soil_moisture_3_to_9cm",
    "9-27 cm": "soil_moisture_9_to_27cm",
    "27-81 cm": "soil_moisture_27_to_81cm",
}

soil_temperature_fields = {
    "surface": "soil_temperature_0cm",
    "6 cm": "soil_temperature_6cm",
    "18 cm": "soil_temperature_18cm",
    "54 cm": "soil_temperature_54cm",
}

soil_moisture_profile = pd.DataFrame([
    {
        "depth": depth,
        "soil_moisture": current_values.get(field),
        "units": current_units.get(field),
    }
    for depth, field in soil_moisture_fields.items()
]).set_index("depth")

soil_temperature_profile = pd.DataFrame([
    {
        "depth": depth,
        "soil_temperature": current_values.get(field),
        "units": current_units.get(field),
    }
    for depth, field in soil_temperature_fields.items()
]).set_index("depth")

print("Current local model time:", current_values.get("time"))
print("\nModeled soil moisture:")
print(soil_moisture_profile)

print("\nModeled soil temperature:")
soil_temperature_profile

Current local model time: 2026-08-26T16:45

Modeled soil moisture:
          soil_moisture  units
depth                         
0-1 cm            0.055  m³/m³
1-3 cm            0.064  m³/m³
3-9 cm            0.095  m³/m³
9-27 cm           0.134  m³/m³
27-81 cm          0.218  m³/m³

Modeled soil temperature:


,soil_temperature,units
depth,,
surface,37.7,°C
6 cm,36.5,°C
18 cm,30.2,°C
54 cm,28.5,°C


## 4. Recent rainfall and seven-day forecast

In [7]:
daily_values = forecast_data["daily"]

daily_df = pd.DataFrame({
    key: value
    for key, value in daily_values.items()
    if key != "time"
})

daily_df.index = pd.to_datetime(daily_values["time"])
daily_df.index.name = "date"
daily_df = daily_df.apply(pd.to_numeric, errors="coerce")

current_local_time = pd.Timestamp(current_values["time"])
current_local_date = current_local_time.normalize()

recent_daily_df = daily_df[
    daily_df.index < current_local_date
].tail(PAST_DAYS)

forecast_daily_df = daily_df[
    daily_df.index >= current_local_date
].head(FORECAST_DAYS)

daily_quality_report = pd.DataFrame({
    "null_count": daily_df.isna().sum(),
    "row_count": len(daily_df),
})

print("Recent complete days:", len(recent_daily_df))
print("Forecast days including today:", len(forecast_daily_df))
print("\nDaily quality report:")
print(daily_quality_report)

forecast_daily_df

Recent complete days: 7
Forecast days including today: 7

Daily quality report:
                               null_count  row_count
temperature_2m_max                      0         14
temperature_2m_min                      0         14
precipitation_sum                       0         14
rain_sum                                0         14
precipitation_probability_max           0         14
et0_fao_evapotranspiration              0         14


,temperature_2m_max,temperature_2m_min,precipitation_sum,rain_sum,precipitation_probability_max,et0_fao_evapotranspiration
date,,,,,,
2026-08-26,35.0,19.4,0.0,0.0,18,6.88
2026-08-27,31.6,20.9,0.0,0.0,15,6.05
2026-08-28,36.4,17.2,0.0,0.0,7,7.00
2026-08-29,37.7,22.9,0.0,0.0,2,9.65
2026-08-30,37.3,22.6,0.0,0.0,0,10.06
2026-08-31,36.8,21.7,0.0,0.0,10,9.46
2026-09-01,36.0,23.2,0.0,0.0,10,9.36


In [8]:
forecast_daily_df = forecast_daily_df.copy()

forecast_daily_df["RAIN_MINUS_ET0_MM"] = (
    forecast_daily_df["precipitation_sum"]
    - forecast_daily_df["et0_fao_evapotranspiration"]
)

operational_summary = pd.Series({
    "recent_7_day_rainfall_mm":
        recent_daily_df["precipitation_sum"].sum(min_count=1),
    "forecast_7_day_rainfall_mm":
        forecast_daily_df["precipitation_sum"].sum(min_count=1),
    "forecast_7_day_et0_mm":
        forecast_daily_df["et0_fao_evapotranspiration"].sum(min_count=1),
    "forecast_rain_minus_et0_mm":
        forecast_daily_df["RAIN_MINUS_ET0_MM"].sum(min_count=1),
    "forecast_wet_days":
        int((forecast_daily_df["precipitation_sum"] >= 1.0).sum()),
    "forecast_heat_days_at_or_above_35c":
        int((forecast_daily_df["temperature_2m_max"] >= 35.0).sum()),
    "forecast_frost_risk_days_at_or_below_0c":
        int((forecast_daily_df["temperature_2m_min"] <= 0.0).sum()),
    "forecast_min_temperature_c":
        forecast_daily_df["temperature_2m_min"].min(),
    "forecast_max_temperature_c":
        forecast_daily_df["temperature_2m_max"].max(),
}).round(2)

print("Operational summary:")
print(operational_summary)

forecast_daily_df.round(2)

Operational summary:
recent_7_day_rainfall_mm                    0.50
forecast_7_day_rainfall_mm                  0.00
forecast_7_day_et0_mm                      58.46
forecast_rain_minus_et0_mm                -58.46
forecast_wet_days                           0.00
forecast_heat_days_at_or_above_35c          6.00
forecast_frost_risk_days_at_or_below_0c     0.00
forecast_min_temperature_c                 17.20
forecast_max_temperature_c                 37.70
dtype: float64


,temperature_2m_max,temperature_2m_min,precipitation_sum,rain_sum,precipitation_probability_max,et0_fao_evapotranspiration,RAIN_MINUS_ET0_MM
date,,,,,,,
2026-08-26,35.0,19.4,0.0,0.0,18,6.88,-6.88
2026-08-27,31.6,20.9,0.0,0.0,15,6.05,-6.05
2026-08-28,36.4,17.2,0.0,0.0,7,7.00,-7.00
2026-08-29,37.7,22.9,0.0,0.0,2,9.65,-9.65
2026-08-30,37.3,22.6,0.0,0.0,0,10.06,-10.06
2026-08-31,36.8,21.7,0.0,0.0,10,9.46,-9.46
2026-09-01,36.0,23.2,0.0,0.0,10,9.36,-9.36


## 5. Hourly validation

The hourly table is retained for later alert logic. Only a preview and missing-value report are shown.

In [9]:
hourly_values = forecast_data["hourly"]

hourly_df = pd.DataFrame({
    key: value
    for key, value in hourly_values.items()
    if key != "time"
})

hourly_df.index = pd.to_datetime(hourly_values["time"])
hourly_df.index.name = "local_time"
hourly_df = hourly_df.apply(pd.to_numeric, errors="coerce")

hourly_quality_report = pd.DataFrame({
    "null_count": hourly_df.isna().sum(),
    "row_count": len(hourly_df),
})

print("Hourly rows:", len(hourly_df))
print("\nHourly quality report:")
print(hourly_quality_report)

hourly_df.head(12)

Hourly rows: 336

Hourly quality report:
                            null_count  row_count
temperature_2m                       0        336
relative_humidity_2m                 0        336
dew_point_2m                         0        336
precipitation                        0        336
rain                                 0        336
precipitation_probability            0        336
et0_fao_evapotranspiration           0        336
vapour_pressure_deficit              0        336
soil_temperature_0cm                 0        336
soil_temperature_6cm                 0        336
soil_temperature_18cm                0        336
soil_temperature_54cm                0        336
soil_moisture_0_to_1cm               0        336
soil_moisture_1_to_3cm               0        336
soil_moisture_3_to_9cm               0        336
soil_moisture_9_to_27cm              0        336
soil_moisture_27_to_81cm             0        336


,temperature_2m,relative_humidity_2m,dew_point_2m,precipitation,rain,precipitation_probability,et0_fao_evapotranspiration,vapour_pressure_deficit,soil_temperature_0cm,soil_temperature_6cm,soil_temperature_18cm,soil_temperature_54cm,soil_moisture_0_to_1cm,soil_moisture_1_to_3cm,soil_moisture_3_to_9cm,soil_moisture_9_to_27cm,soil_moisture_27_to_81cm
local_time,,,,,,,,,,,,,,,,,
2026-08-19 00:00:00,25.4,33,7.9,0.0,0.0,0,0.11,2.17,23.7,28.8,30.5,28.1,0.043,0.072,0.104,0.141,0.222
2026-08-19 01:00:00,24.8,36,8.7,0.0,0.0,0,0.10,2.00,23.4,28.2,30.3,28.1,0.044,0.072,0.104,0.141,0.222
2026-08-19 02:00:00,23.8,39,9.0,0.0,0.0,0,0.09,1.80,22.8,27.7,30.1,28.1,0.044,0.072,0.104,0.141,0.222
2026-08-19 03:00:00,23.3,36,7.4,0.0,0.0,0,0.10,1.83,22.2,27.2,29.8,28.1,0.044,0.072,0.104,0.141,0.222
2026-08-19 04:00:00,21.9,42,8.4,0.0,0.0,0,0.10,1.52,21.6,26.7,29.5,28.1,0.044,0.072,0.104,0.141,0.222
2026-08-19 05:00:00,22.9,42,9.3,0.0,0.0,0,0.13,1.62,21.3,26.3,29.3,28.1,0.045,0.072,0.104,0.142,0.222
2026-08-19 06:00:00,22.2,44,9.3,0.0,0.0,0,0.10,1.49,20.7,25.9,29.0,28.1,0.045,0.071,0.104,0.142,0.222
2026-08-19 07:00:00,21.2,44,8.5,0.0,0.0,0,0.08,1.41,19.7,25.3,28.7,28.1,0.045,0.071,0.104,0.142,0.221
2026-08-19 08:00:00,21.9,44,9.1,0.0,0.0,0,0.08,1.47,21.3,25.0,28.4,28.1,0.045,0.071,0.104,0.142,0.221


## 6. Expected validation-error test

In [10]:
invalid_params = {
    "latitude": FARM_LATITUDE,
    "longitude": FARM_LONGITUDE,
    "hourly": "NOT_A_REAL_VARIABLE",
    "forecast_days": 1,
    "timezone": "auto",
}

invalid_response = requests.get(
    FORECAST_URL,
    params=invalid_params,
    timeout=TIMEOUT_SECONDS,
)

print("HTTP status:", invalid_response.status_code)

try:
    print(invalid_response.json())
except ValueError:
    print(invalid_response.text)

HTTP status: 400
{'error': True, 'reason': "Data corrupted at path ''. Cannot initialize SurfacePressureAndHeightVariable<VariableAndPreviousDay, VariableOrSpread<ForecastPressureVariable>, ForecastHeightVariable> from invalid String value NOT_A_REAL_VARIABLE."}


In [ ]:
import json
from pathlib import Path

# Export the validated Open-Meteo evidence for the bundle builder.
OUTPUT_PATH = Path(
    "data/evidence/providers/open_meteo_plainview.json"
)
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

def json_default(value):
    if isinstance(value, pd.Timestamp):
        return value.isoformat()
    if hasattr(value, "item"):
        return value.item()
    raise TypeError(
        f"Object of type {type(value).__name__} is not JSON serializable"
    )

def dataframe_records(frame):
    export = frame.reset_index().copy()
    for column in export.columns:
        if pd.api.types.is_datetime64_any_dtype(export[column]):
            export[column] = export[column].dt.strftime(
                "%Y-%m-%dT%H:%M:%S"
            )
    export = export.astype(object).where(pd.notna(export), None)
    return export.to_dict(orient="records")

def series_values(series):
    return {
        key: (None if pd.isna(value) else value)
        for key, value in series.items()
    }

daily_null_counts = {
    key: int(value)
    for key, value in daily_quality_report["null_count"].items()
}
hourly_null_counts = {
    key: int(value)
    for key, value in hourly_quality_report["null_count"].items()
}
current_missing_fields = [
    field
    for field in CURRENT_VARIABLES
    if field not in current_values or current_values.get(field) is None
]

open_meteo_evidence = {
    "schema_version": "1.0",
    "provider": "Open-Meteo",
    "status": "validated",
    "generated_at": pd.Timestamp.now(tz="UTC").isoformat(),
    "farm": {
        "farm_id": "plainview_demo",
        "farm_name": "Plainview demonstration farm",
        "latitude": FARM_LATITUDE,
        "longitude": FARM_LONGITUDE,
        "crop_region": "Texas Plains",
        "timezone": forecast_data.get("timezone"),
    },
    "interpretation": {
        "evidence_role": "recent_and_forecast_weather",
        "spatial_warning": (
            "Modeled weather-grid evidence; not an on-farm sensor "
            "measurement."
        ),
        "soil_warning": (
            "Modeled soil moisture and temperature; permanent soil "
            "properties must come from SSURGO or a soil test."
        ),
        "water_warning": (
            "Rain minus ET0 is a screening indicator, not an "
            "irrigation prescription."
        ),
    },
    "request_trace": {
        "geocoding_url": geocoding_response.url,
        "forecast_url": forecast_response.url,
        "invalid_parameter_status": invalid_response.status_code,
    },
    "geocoding": {
        "query": LOCATION_QUERY,
        "selected_result": series_values(geocoding_summary),
        "limitation": (
            "City-level lookup only; the weather request uses the "
            "farm coordinate."
        ),
    },
    "provider_grid": {
        "latitude": forecast_data.get("latitude"),
        "longitude": forecast_data.get("longitude"),
        "elevation_m": forecast_data.get("elevation"),
        "timezone": forecast_data.get("timezone"),
        "timezone_abbreviation": (
            forecast_data.get("timezone_abbreviation")
        ),
        "utc_offset_seconds": (
            forecast_data.get("utc_offset_seconds")
        ),
    },
    "units": {
        "current": forecast_data.get("current_units", {}),
        "hourly": forecast_data.get("hourly_units", {}),
        "daily": forecast_data.get("daily_units", {}),
    },
    "current": {
        "model_time": current_values.get("time"),
        "values": current_values,
        "soil_moisture_profile": dataframe_records(
            soil_moisture_profile
        ),
        "soil_temperature_profile": dataframe_records(
            soil_temperature_profile
        ),
    },
    "recent_complete_days": {
        "requested_days": PAST_DAYS,
        "records": dataframe_records(recent_daily_df),
    },
    "forecast": {
        "requested_days": FORECAST_DAYS,
        "daily_records": dataframe_records(forecast_daily_df),
        "hourly_records": dataframe_records(hourly_df),
        "operational_summary": series_values(
            operational_summary
        ),
    },
    "quality": {
        "current_missing_fields": current_missing_fields,
        "daily_null_counts": daily_null_counts,
        "hourly_null_counts": hourly_null_counts,
        "recent_day_count": len(recent_daily_df),
        "forecast_day_count": len(forecast_daily_df),
        "hourly_record_count": len(hourly_df),
    },
}

OUTPUT_PATH.write_text(
    json.dumps(
        open_meteo_evidence,
        indent=2,
        allow_nan=False,
        default=json_default,
    ),
    encoding="utf-8",
)

print("Saved normalized provider evidence:")
print(OUTPUT_PATH.resolve())
print("Recent complete days:", len(recent_daily_df))
print("Forecast days:", len(forecast_daily_df))
print("Hourly records:", len(hourly_df))
print("Timezone:", forecast_data.get("timezone"))

# Final Open-Meteo Integration Decision

## Selected responsibilities

CropSage may use Open-Meteo for:

- modeled current soil moisture and soil temperature by depth;
- recent rainfall totals and dry-spell context;
- short-term precipitation, temperature, ET₀ and VPD forecasts;
- frost, heat and wet-day operational flags;
- automatic IANA time-zone resolution.

## Responsibilities it does not replace

- NASA POWER remains the long-term regional climate baseline.
- FortyGuard remains the required local heat-intelligence source.
- SSURGO supplies permanent soil properties and available water storage.
- A dedicated address geocoder or map pin is required for exact farms.
- NOAA normals or a validated historical calculation is required for frost-free season.
- The crop catalog must supply biological root-zone depth.

## Interpretation limits

Open-Meteo values are model estimates for a weather grid cell, not on-farm sensor measurements. The rain-minus-ET₀ calculation is a screening indicator, not a soil-water balance or irrigation prescription. Soil moisture must be clearly labeled as modeled and should be combined with SSURGO, recent weather, irrigation access and farmer observations.